# Part 4: Evaluation

In [1]:
import gc
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from src.baseline_features import train_logreg, evaluate_model
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

## Step 1: Retrain Baseline Model on Full Training Data
We retrain the baseline model using the same hyperparameters as in Part 2.

In [2]:
# Part 4 - Load train data
TRAIN_PATH = '../data/processed/news_stratified_train.csv'

train_df = pd.read_csv(
    TRAIN_PATH,
    usecols=['content_processed', 'type'],
    on_bad_lines='skip',
    dtype={'type': 'int8'},
    nrows=300000 # Memory issues so we limit the number of rows for now
)
train_df = train_df.dropna(subset=['content_processed', 'type']).copy()
train_df['type'] = train_df['type'].astype(int)

print(f"Train: {train_df.shape}")
print(f"Label distribution:\n{train_df['type'].value_counts()}")

Train: (299999, 2)
Label distribution:
type
1    168888
0    131111
Name: count, dtype: int64


In [3]:
# Part 4 - Build features and train baseline model
vectorizer = CountVectorizer(max_features=5000, dtype=np.float32)
X_train = vectorizer.fit_transform(train_df['content_processed'].fillna('').astype(str))

y_train = train_df['type'].values

del train_df
gc.collect()

model = train_logreg(
    X_train,
    y_train,
    C=1.0,
    max_iter=100,
    solver='liblinear',
    class_weight='balanced'
)

del X_train
gc.collect()

print("Model trained.")
print(f"Hyperparameters: C={model.C}, max_iter={model.max_iter}, solver={model.solver}, class_weight={model.class_weight}")

Model trained.
Hyperparameters: C=1.0, max_iter=100, solver=liblinear, class_weight=balanced


## Step 2: Evaluate on FakeNewsCorpus Test Set
We evaluate the baseline model on the held-out test set.

In [4]:
# Part 4 - Evaluate on test set
TEST_PATH = '../data/processed/news_stratified_test.csv'

test_df = pd.read_csv(
    TEST_PATH,
    usecols=['content_processed', 'type'],
    on_bad_lines='skip',
    dtype={'type': 'int8'}
)
test_df = test_df.dropna(subset=['content_processed', 'type']).copy()
test_df['type'] = test_df['type'].astype(int)

X_test = vectorizer.transform(test_df['content_processed'].fillna('').astype(str))
y_test = test_df['type'].values

del test_df
gc.collect()

results_test = evaluate_model(model, X_test, y_test, label_names=['reliable', 'fake'])
print(results_test['report'])
print(f"Macro F1: {results_test['macro_f1']:.4f}")
print("\nConfusion matrix:")
print(results_test['confusion_matrix'])

              precision    recall  f1-score   support

    reliable       0.83      0.80      0.82     41308
        fake       0.85      0.88      0.86     53414

    accuracy                           0.85     94722
   macro avg       0.84      0.84      0.84     94722
weighted avg       0.84      0.85      0.84     94722

Macro F1: 0.8418

Confusion matrix:
[[33221  8087]
 [ 6587 46827]]


## Step 3: Evaluate on LIAR Dataset
We evaluate the baseline model on the LIAR dataset without retraining. LIAR has 6 labels which we map to binary: true, mostly-true, and half-true become reliable (0), while barely-true, false, and pants-fire become fake (1).

In [5]:
# Part 4 - Load and prepare LIAR test set
LIAR_PATH = '../data/raw/liar/test.tsv'

liar_cols = [0, 1, 2]  # id, label, statement
liar_df = pd.read_csv(
    LIAR_PATH,
    sep='\t',
    header=None,
    usecols=liar_cols,
    names=['id', 'label', 'statement']
)

# Map 6 labels to binary
reliable_labels = {'true', 'mostly-true', 'half-true'}
liar_df['type'] = liar_df['label'].apply(lambda x: 0 if x in reliable_labels else 1)

print(f"LIAR test set: {liar_df.shape}")
print(f"\nOriginal label distribution:\n{liar_df['label'].value_counts()}")
print(f"\nBinary label distribution:\n{liar_df['type'].value_counts()}")

LIAR test set: (1267, 4)

Original label distribution:
label
half-true      265
false          249
mostly-true    241
barely-true    212
true           208
pants-fire      92
Name: count, dtype: int64

Binary label distribution:
type
0    714
1    553
Name: count, dtype: int64


## Step 4: Cross-Domain Evaluation
We apply the preprocessing pipeline to the LIAR statements and evaluate the baseline model directly.

In [6]:
# Part 4 - Preprocess LIAR statements and evaluate
from src.preprocessing import normalize_text
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

stop_words = set(stopwords.words('english'))
stemmer = SnowballStemmer('english')

def process_text(text):
    text = normalize_text(text)
    tokens = text.split()
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words]
    return " ".join(tokens)

liar_df['content_processed'] = liar_df['statement'].apply(process_text)

X_liar = vectorizer.transform(liar_df['content_processed'].fillna('').astype(str))
y_liar = liar_df['type'].values

results_liar = evaluate_model(model, X_liar, y_liar, label_names=['reliable', 'fake'])
print(results_liar['report'])
print(f"Macro F1: {results_liar['macro_f1']:.4f}")
print("\nConfusion matrix:")
print(results_liar['confusion_matrix'])

              precision    recall  f1-score   support

    reliable       0.61      0.08      0.14       714
        fake       0.44      0.93      0.60       553

    accuracy                           0.45      1267
   macro avg       0.52      0.51      0.37      1267
weighted avg       0.54      0.45      0.34      1267

Macro F1: 0.3687

Confusion matrix:
[[ 56 658]
 [ 36 517]]
